# Scraping website using Selenium

### Importing modules

In [ ]:
import os 
from dotenv import load_dotenv
from openai import OpenAI
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

### Verify API Key & Connect to OpenAI

In [ ]:

load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("No API Key was found.")
elif not api_key.startswith("sk-proj-"):
    print("An API Key was found, but it didn't startefd with 'sk-proj-'")
elif api_key.strip() != api_key:
    print("An API Key was found, but it has some extra spaces or tabs in the characters.")
else:
    print("An API Key was found and looks good so far!")

openai_client = OpenAI()

An API Key was found and looks good so far!


### Setting system_prompt and user_prompt

In [ ]:
system_prompt = """
You are an snarky assistant that analyzes the contents of the website, 
and provides a short, snarky, humouros summary, ignoring text that might be
navigation related. Respond in markdown. Do not wrap the markdown in a code 
block - respond just with the markdown.
"""

user_prompt_prefix  = """
Here are the contents of the website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

WEBSITE CONTENT:
"""

### Fetching the contents of website

In [ ]:
def fetch_website_contents(url):
    print(f"Fetching content from {url} using Selenium...")
    driver = None

    try:
        print("1. Setting up Chrome options....")
        chrome_options = Options()
        chrome_options.add_argument("--headless")
        chrome_options.add_argument("--disable-gpu")
        chrome_options.add_argument("window-size=1200x600")
        chrome_options.add_argument("--no-sandbox")
        chrome_options.add_argument("--disable-dev-shm-usage")

        print("2. Installing/Finding WebDriver....")
        driver_path = ChromeDriverManager().install()
        print(f"WebDriver is at: {driver_path}")
        service = Service(driver_path)

        print("3. Initializing WebDriver....")
        driver = webdriver.Chrome(service=service, options=chrome_options)

        print("4. Setting page load timeout...")
        driver.set_page_load_timeout(30)

        print(f"5. Getting URL: {url}....")
        driver.get(url)

        print("6. Setting implicit wait...")
        driver.implicitly_wait(5)

        print("7. Finding body content...")
        body_content = driver.find_element(By.TAG_NAME, "body").text

        # Use the below code for heavy sites
        # print("6. Waiting for body content to load...")
        # element = WebDriverWait(driver, 20).until(
        #     EC.presence_of_element_located((By.TAG_NAME, "body"))
        # )
        # body_content = element.text        

        print("Content fetched successfully.")
        print(f"Fetched content length: {len(body_content)}")
        return body_content

    except Exception as e:
        print(f"Error during Selenium operation: {e}")
        if driver:
            body_content = driver.find_element(By.TAG_NAME, "body").text

            if body_content:
                print("Page timed out, but returning partial content.")
                return body_content
            
        return None
    
    finally:
        if driver:
            print("8. Shutting down WebDriver....")
            driver.close()
            driver.quit()
            print("WebDriver Shut Down.")

### Creating Summarize function

In [ ]:

def summarize(url):
    website_content = fetch_website_contents(url)
    
    if not website_content:
        return f"Sorry, I couldn't fetch the website. It's probably broken or my robot eyes can't see it."

    messages = [
    {'role': 'system', 'content': system_prompt},
    {'role': 'user', 'content': user_prompt_prefix + website_content}
    ]

    try:
        print("9. Calling OpenAI API....")
        response = openai_client.chat.completions.create(
            model= "gpt-4-turbo",
            messages= messages
        )

        print("OpenAI call successful...")
        return response.choices[0].message.content

    except Exception as e:
        print(f"Error calling OpenAI: {e}")
        return "Error from AI: It probably got bored and left."


In [ ]:
target_url = "http://example.com"
snarky_summary = summarize(target_url)

print("\n----SNARKY SUMMARY-------")
print(snarky_summary)